# ExhibitFlow — backend cho web (Kaggle)

Chạy **engine của dashboard** (model nạp sẵn một lần, chạy nhiều video) và mở **URL công khai** (Cloudflare) để **web giao diện đen** (`web/`) kết nối vào.

**Trước khi chạy:** *Accelerator* → **GPU T4 x2**; *Internet* → **On**; *Add Input* → dataset chứa **4 file trọng số** (`best_ckpt.pth`, `osnet_x0_25.pth`, `ltc_motion_mot17.pth`, `dma_gbm_mot17.gbm`).

**Cách dùng:** chạy lần lượt các ô. Ô cuối nạp model (lần đầu vài phút) rồi in **URL BACKEND**. Mở web (`web/`) → **Settings → URL backend** dán URL đó → **Kết nối** → Dashboard → Upload video → **Phân tích**. Giữ ô cuối chạy suốt lúc demo.

## 1. Cấu hình

In [ ]:
REPO_URL = 'https://github.com/khangkaka066/ExhibitFlow.git'
BRANCH   = 'Thuan'   # nhánh chứa server/web_server.py (đổi nếu bạn push lên nhánh khác)
ROOT     = '/kaggle/working/ExhibitFlow'
WEIGHTS  = ''      # thư mục chứa 4 file trọng số; để trống = tự tìm trong /kaggle/input
DEMO     = False   # True = dữ liệu giả để thử giao diện (không cần GPU / trọng số)
PORT     = 8000
HF_TOKEN = ''      # KHÔNG ghi token vào notebook rồi push. Để trống: ô cuối sẽ hỏi lúc chạy (không hiện ký tự)

## 2. Lấy mã nguồn (+ map 2D)

In [ ]:
import os, subprocess
def sh(c, **k):
    print('$', c); subprocess.run(c, shell=True, check=True, **k)
if not os.path.exists(ROOT):
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {ROOT}')
else:
    sh(f'cd {ROOT} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD')
FP = os.path.join(os.path.dirname(ROOT), 'multicam-2d-floorplan-tracker')
if not os.path.exists(FP):
    sh(f'git clone --depth 1 https://github.com/khangkaka066/multicam-2d-floorplan-tracker.git {FP}')
os.environ['FLOORPLAN_DIR'] = FP
PROJ = os.path.join(ROOT, 'exhibitflow_pipeline')
assert os.path.exists(os.path.join(PROJ, 'server', 'web_server.py')),     'Không thấy server/web_server.py: nhánh ' + BRANCH + ' chưa có file này (cần push lên GitHub trước).'
print('OK', PROJ)

## 3. Cài thư viện (~3-5 phút lần đầu)

In [ ]:
sh(f'cd {PROJ} && python scripts/install_deps.py')
sh('pip install -q fastapi "uvicorn[standard]" python-multipart psutil')

## 4. Tìm trọng số

In [ ]:
import glob
if not WEIGHTS:
    hits = glob.glob('/kaggle/input/**/best_ckpt.pth', recursive=True)
    WEIGHTS = os.path.dirname(hits[0]) if hits else ''
print('Thư mục trọng số:', WEIGHTS or '(không thấy: thêm dataset trọng số ở Add Input, hoặc đặt WEIGHTS ở ô 1)')
if not DEMO and not WEIGHTS:
    print('!!! Chưa có trọng số, model sẽ không nạp được.')

## 5. Chạy backend + mở URL công khai (giữ ô này chạy)

In [ ]:
import getpass, sys
subprocess.run(['pkill', '-f', 'web_server.py'])      # dọn server / tunnel còn sót từ lần chạy trước
subprocess.run(['pkill', '-f', 'cloudflared'])
env = dict(os.environ)
if WEIGHTS: env['EF_WEIGHTS_DIR'] = WEIGHTS
token = ''
if not DEMO:
    token = HF_TOKEN or os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN (Enter để bỏ qua, tải sẽ chậm hơn): ').strip()
if token:
    env['HF_TOKEN'] = token            # huggingface_hub tự đọc biến này khi tải GGUF / Grounding-DINO
args = [sys.executable, '-u', 'server/web_launch.py', '--port', str(PORT), '--data-dir', os.path.join(PROJ, 'dashboard_data')] + (['--demo'] if DEMO else [])
subprocess.run(args, cwd=PROJ, env=env)